
# PR1 — Limpieza, EDA y visualización de *logs* de seguridad

**Estudiante:** _Astudillo Soriano, Johnny_  

**Fecha:** _2026-11-01_  

**Dataset:** `data/PR1_security_logs_Mar2025.csv`


## Objetivos y entregables

- Tratar datos de ciberseguridad: **ingesta → limpieza → normalización → EDA → visualización → detección de campañas → informe**.
- Usar `pandas`/`numpy` y graficar con `matplotlib` o `plotly`.
- Entregar:
  - `notebook/PR1_<apellidos>_<nombre>.ipynb` (este cuaderno completado)
  - `informe/informe_PR1_<apellidos>_<nombre>.pdf` (máx. 5 páginas)
  - `resultados/figuras/*.png|.html`
  - `resultados/eventos_flagged.csv`
  - `data/PR1_security_logs_Mar2025.csv`
  - `requirements.txt` o `environment.yml`


In [202]:

# === Configuración de entorno y rutas ===
from pathlib import Path

# Rutas base del proyecto (asume estructura de la plantilla)
BASE = Path().resolve() / ".."
DATA = BASE / "data" / "PR1_security_logs_Mar2025.csv"
OUT_DIR = BASE / "resultados"
FIG_DIR = OUT_DIR / "figuras"

# Crea carpetas de salida si no existen
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Semilla opcional (para reproducibilidad en muestreos, etc.)
SEED = 42

print("BASE:", BASE)
print("DATA:", DATA)
print("OUT_DIR:", OUT_DIR)
print("FIG_DIR:", FIG_DIR)


BASE: /Users/uoc/Developer/PR1_Astudillo_Soriano_Johnny/notebook/..
DATA: /Users/uoc/Developer/PR1_Astudillo_Soriano_Johnny/notebook/../data/PR1_security_logs_Mar2025.csv
OUT_DIR: /Users/uoc/Developer/PR1_Astudillo_Soriano_Johnny/notebook/../resultados
FIG_DIR: /Users/uoc/Developer/PR1_Astudillo_Soriano_Johnny/notebook/../resultados/figuras


In [203]:

# === Global Imports ===
import re
import ipaddress
from datetime import datetime, timezone
import tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [204]:
pd.set_option('display.max_columns', None)

In [205]:
# === Colors for enhanced outputs ===
GREEN = "\033[0;32m"
GREEN_BOLD = "\033[1;32m"
BOLD = "\033[1m"
RESET = "\033[0m"

In [206]:
def save_figure(fig_name: str) -> None:
    """
    Saves the active figure in results/figures.

    Parameters
    ----------
    fig_name : str
        Name of the figure file to save.
    """
    assert fig_name, "Especifica un nombre para la figura, p.ej. 'req_por_dia.png'"
    path = FIG_DIR / fig_name
    plt.savefig(path, bbox_inches="tight", dpi=120)
    print("Figura guardada en:", path)



---
## Etapa A — Ingesta y *profiling* inicial

**Tareas:**  
- Carga el CSV en un `DataFrame` `df`.  
- Explora `shape`, `info()`, `describe()`, ejemplos de filas (`head()`, `tail()`).  
- Identifica tipos esperados vs reales; localiza nulos y valores inválidos (timestamps imposibles, IPs malformadas, `bytes_sent` vacíos).  
- Busca duplicados relevantes (decide la clave: ¿`event_id` único? ¿tupla clave?).

**Recomendaciones:**  
- Usa `pd.read_csv(..., low_memory=False)` si ves *dtypes* mezclados.  
- Declara `na_values=["", "-"]` si procede.  
- Documenta DECISIONES de limpieza que anticipas.


### Dataset Loading

In [207]:
# === Data Loading ===
df = pd.read_csv(DATA, low_memory=False)

### Dataset Profiling

In [208]:
print(f"{GREEN_BOLD}Dataset shape:{RESET} {df.shape}")

Dataset shape: (11775, 24)


In [209]:
print(f"{GREEN_BOLD}General info:{RESET}")
print(df.info(verbose=True, memory_usage=False, show_counts=True))

General info:
<class 'pandas.DataFrame'>
RangeIndex: 11775 entries, 0 to 11774
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   event_id          11775 non-null  int64  
 1   timestamp         11744 non-null  str    
 2   src_ip            11775 non-null  str    
 3   src_port          11775 non-null  int64  
 4   src_country       11775 non-null  str    
 5   dest_ip           11775 non-null  str    
 6   dest_port         11775 non-null  int64  
 7   host              11775 non-null  str    
 8   http_method       11775 non-null  str    
 9   url_path          11775 non-null  str    
 10  query_string      2984 non-null   str    
 11  status            11775 non-null  int64  
 12  bytes_sent        11394 non-null  float64
 13  referrer          9069 non-null   str    
 14  user_agent        11775 non-null  str    
 15  protocol          11775 non-null  str    
 16  http_version      11775 non-null  flo

In [210]:
print(f"First {GREEN_BOLD}3{RESET} rows:")
df.head(3)

First 3 rows:


,event_id,timestamp,src_ip,src_port,src_country,dest_ip,dest_port,host,http_method,url_path,query_string,status,bytes_sent,referrer,user_agent,protocol,http_version,tls_version,response_time_ms,event_source,waf_action,waf_rule_id,session_id,auth
0,3408,2025-03-10T10:26:00+00:00,198.51.100.124,13054,US,198.51.100.36,80,api.kaluma.io,POST,/cart,NaN,404,442.0,https://twitter.com/ref,Mozilla/5.0 (iPhone; CPU iPhone OS 16_6 like M...,HTTP/1.1,1.1,NaN,22.0,apache,NaN,NaN,sid-706b94500819,user
1,1474,2025-03-05T10:26:08+00:00,198.51.100.87,64430,PL,198.51.100.190,443,cdn.sentryargo.org,GET,/images/logo.png,NaN,200,91252.0,https://news.ycombinator.com,Mozilla/5.0 (Linux; Android 14; Pixel 8) Apple...,HTTP/3,1.1,TLS1.3,59.0,apache,NaN,NaN,sid-b4bdc36b9a6d,anonymous
2,9447,2025-03-26T14:31:37+00:00,198.51.100.87,25347,CN,198.51.100.36,80,m.kaluma.io,GET,/,NaN,304,718.0,https://omnisynargo.net/previous,Mozilla/5.0 (iPhone; CPU iPhone OS 16_6 like M...,HTTP/1.1,1.1,NaN,212.0,nginx,NaN,NaN,sid-2a79f7a16855,anonymous


In [211]:
print(f"Last {GREEN_BOLD}3{RESET} rows:")
df.tail(3)

Last 3 rows:


,event_id,timestamp,src_ip,src_port,src_country,dest_ip,dest_port,host,http_method,url_path,query_string,status,bytes_sent,referrer,user_agent,protocol,http_version,tls_version,response_time_ms,event_source,waf_action,waf_rule_id,session_id,auth
11772,2284,2025-03-07T14:50:10+00:00,203.0.113.80,38080,IE,198.51.100.214,443,cdn.liforge.dev,GET,/checkout,NaN,301,312.0,NaN,Mozilla/5.0 (Macintosh; Intel Mac OS X 13_5) A...,HTTP/3,1.1,TLS1.3,668.0,apache,NaN,NaN,sid-0579e27a9eb1,anonymous
11773,10197,28/Mar/2025:02:06:55 +0000,203.0.113.138,33052,ES,203.0.113.192,443,cdn.kaluma.io,GET,/favicon.ico,NaN,304,282.0,https://twitter.com/ref,Mozilla/5.0 (iPhone; CPU iPhone OS 16_6 like M...,HTTP/2,1.1,TLS1.3,845.0,nginx,NaN,NaN,NaN,-
11774,8800,2025-03-24T03:05:09+00:00,203.0.113.250,36667,IN,192.0.2.74,443,api.omnisynargo.net,HEAD,/.env,NaN,404,526.0,https://news.ycombinator.com,wpscan/3.8.25 (http://wpscan.com/wordpress-sec...,HTTP/2,1.1,TLS1.3,115.0,waf,NaN,NaN,NaN,-


In [212]:
print(f"{GREEN_BOLD}response_time_ms{RESET} and {GREEN_BOLD}bytes_sent{RESET} statistics:")
df[["response_time_ms", "bytes_sent"]].describe()

response_time_ms and bytes_sent statistics:


,response_time_ms,bytes_sent
count,11775.00000,11394.000000
mean,423.24000,13187.708355
std,256.38381,35642.135164
min,10.00000,150.000000
25%,199.50000,446.000000
50%,389.00000,715.500000
75%,644.00000,7136.500000
max,899.00000,199961.000000


### Dataset Quality Analysis

In [248]:
# === Null values ===
null_values = df.isna()
total_null = null_values.sum()
null_proportion = (null_values.mean() * 100).round(2)

null_summary = pd.DataFrame({
    'dtype': df.dtypes,
    'count': total_null,
    '%': null_proportion,
})

print(f"{GREEN_BOLD}Null values:{RESET}")
null_summary

Null values:


,dtype,count,%
event_id,int64,0,0.00
timestamp,str,31,0.26
src_ip,str,0,0.00
src_port,int64,0,0.00
src_country,str,0,0.00
dest_ip,str,0,0.00
dest_port,int64,0,0.00
host,str,0,0.00
http_method,str,0,0.00
url_path,str,0,0.00


In [245]:
# === Empty strings ===
empty_strings = df.select_dtypes(include=['object', 'string']).map(lambda x: str(x).strip() == '')
total_empty_strings = empty_strings.sum()
empty_string_proportion = (empty_strings.mean() * 100).round(2)

empty_string_summary = pd.DataFrame({
    'count': total_empty_strings,
    '%': empty_string_proportion,
})

print(f"{GREEN_BOLD}Empty strings:{RESET}")
empty_string_summary

Empty strings:


,count,%
timestamp,0,0.0
src_ip,0,0.0
src_country,0,0.0
dest_ip,0,0.0
host,0,0.0
http_method,0,0.0
url_path,0,0.0
query_string,0,0.0
referrer,0,0.0
user_agent,0,0.0


In [ ]:
# === Malformed IPs ===
IP_REGEX = re.compile(r'\b\d{1,4}\.\d{1,4}\.\d{1,4}\.\d{1,4}\b')

malformed_ips = df[["src_ip", "dest_ip"]].map(lambda x: not bool(IP_REGEX.match(str(x).strip())))
total_malformed_ips = malformed_ips.sum()
malformed_ip_proportion = (malformed_ips.mean() * 100).round(2)

malformed_ips_summary = pd.DataFrame({
    'count': total_malformed_ips,
    '%': malformed_ip_proportion,
})

print(f"{GREEN_BOLD}Malformed IPs:{RESET}")
malformed_ips_summary

Malformed IPs:


,count,%
src_ip,1181,10.03
dest_ip,0,0.00


In [ ]:
# === Invalid Ports ===
invalid_ports = df[["src_port", "dest_port"]].map(lambda x: not (0 <= x <= 65535))
total_invalid_ports = invalid_ports.sum()
invalid_port_proportion = (invalid_ports.mean() * 100).round(2)

invalid_ports_summary = pd.DataFrame({
    'count': total_invalid_ports,
    '%': invalid_port_proportion,
})

print(f"{GREEN_BOLD}Invalid Ports:{RESET}")
invalid_ports_summary

Invalid Ports:


,count,%
src_port,11775,100.0
dest_port,11775,100.0


In [253]:
df["host"].map(lambda x: not bool(DOMAIN_REGEX.match(str(x).strip())))

0        False
1        False
2        False
3        False
4        False
         ...  
11770    False
11771    False
11772    False
11773    False
11774    False
Name: host, Length: 11775, dtype: bool

In [ ]:
# === Invalid Hosts ===
DOMAIN_REGEX = re.compile(r'^(?:[a-zA-Z0-9](?:[a-zA-Z0-9-]{0,61}[a-zA-Z0-9])?\.)+[a-zA-Z]{2,}$')
invalid_hosts = df["host"].map(lambda x: not bool(DOMAIN_REGEX.match(str(x).strip())))
total_invalid_hosts = invalid_hosts.sum()
invalid_host_proportion = (invalid_hosts.mean() * 100).round(2)

invalid_hosts_summary = pd.DataFrame({
    'count': total_invalid_hosts,
    '%': invalid_host_proportion,
})

print(f"{GREEN_BOLD}Invalid Hosts:{RESET}")
invalid_hosts_summary

ValueError: If using all scalar values, you must pass an index


---
## Etapa B — Limpieza y normalización

**Objetivo:** crear columnas limpias y derivadas para facilitar el análisis.

**Requerido (nuevo esquema):**
- `ts` (datetime UTC) a partir de `timestamp` con parseo robusto.
- `http_method_norm` (método en mayúsculas).
- `bytes_sent_num` (numérico, decide tratamiento de vacíos).
- `status_class` ∈ {`2xx`,`3xx`,`4xx`,`5xx`}.
- `src_ip_kind` ∈ {`ipv4`,`ipv6`,`private`,`malformed`} usando `ipaddress`.
- Derivadas: `day`, `hour`, `weekday`, `is_weekend`, `is_asset` (extensión), `path_depth`, `has_query`, `query_len`.
- `ua_family` ∈ {`browser`,`bot`,`tool`,`rare`} (clasificación sencilla por patrones en `user_agent`).
- (Chequeo) `host_dest_consistent` (consistencia `host`↔`dest_ip`/`dest_port`).

**Pistas:**  
- Prueba primero `pd.to_datetime(..., errors="coerce", utc=True)`; si no basta, considera un *parser* por formatos.  
- `ipaddress.ip_address(x)` + `is_private` para IPs. Maneja `ValueError`.  
- Crea funciones pequeñas y testeables.


In [ ]:

# === Stubs / esqueletos (completa las funciones) ===

def parse_timestamp_series(s: pd.Series) -> pd.Series:
    """Convierte 'timestamp' a datetime UTC robusto.
    Pistas:
      - pd.to_datetime(..., errors='coerce', utc=True)
      - Si detectas múltiples formatos, prueba estrategias secuenciales.
    Devuelve: pd.Series con dtype datetime64[ns, UTC].
    """
    raise NotImplementedError


def classify_ip_value(ip_str: str) -> str:
    """Clasifica IP en {'ipv4','ipv6','private','malformed'}.
    Pistas:
      - ipaddress.ip_address(ip_str)
      - .version, .is_private
      - Maneja ValueError para 'malformed'
    """
    raise NotImplementedError


def normalize_http_method(x: str) -> str:
    """Normaliza método HTTP a mayúsculas; maneja nulos."""
    raise NotImplementedError


def status_to_class(code: int) -> str:
    """Mapea 200-299→'2xx', 300-399→'3xx', 400-499→'4xx', 500-599→'5xx'."""
    raise NotImplementedError


def ua_to_family(ua: str) -> str:
    """Clasifica user-agent en {'browser','bot','tool','rare'} con reglas simples.
    Pistas:
      - patrones como 'Googlebot', 'bingbot', 'DuckDuckBot' → 'bot'
      - 'curl', 'sqlmap', 'wpscan', 'python-requests' → 'tool'
      - patrones de navegadores comunes → 'browser'
      - si no coincide → 'rare'
    """
    raise NotImplementedError


In [ ]:

# TODO: Aplica transformaciones para crear columnas limpias/derivadas.
# df['ts'] = parse_timestamp_series(df['timestamp'])
# df['http_method_norm'] = df['http_method'].apply(normalize_http_method)
# df['bytes_sent_num'] = pd.to_numeric(df['bytes_sent'], errors='coerce')
# df['status_class'] = df['status'].apply(status_to_class)
# df['src_ip_kind'] = df['src_ip'].apply(classify_ip_value)
# df['day'] = df['ts'].dt.date
# df['hour'] = df['ts'].dt.hour
# df['weekday'] = df['ts'].dt.weekday
# df['is_weekend'] = df['weekday'].isin([5,6])
# df['is_asset'] = df['url_path'].str.endswith(('.js','.css','.png','.ico'), na=False)
# df['path_depth'] = df['url_path'].fillna('').str.count('/')
# df['has_query'] = df['query_string'].fillna('') != ''
# df['query_len'] = df['query_string'].fillna('').str.len()
# df['ua_family'] = df['user_agent'].apply(ua_to_family)

# TODO: host_dest_consistent (regla propia y justificación en el informe)
# df['host_dest_consistent'] = ...

# raise NotImplementedError("Completa la normalización y columnas derivadas")



---
## Etapa C — EDA y visualizaciones

**Requerido (guardar figuras en `resultados/figuras/`):**
- Serie temporal diaria (req/día).
- Heatmap hora×día.
- Barras apiladas: `status_class` por `host`, y por `ua_family`.
- Rankings: top IPs por **404**; top `url_path`.
- Histograma de `response_time_ms` (y, si quieres, boxplots por `host`).
- Gráfico libre con *insight* propio.

> Recuerda: con matplotlib, **una figura por gráfico** y evita fijar colores/estilos.


In [ ]:

# TODO: Ejemplo de estructura (NO resuelto)
# --- Req/día ---
# grp = df.groupby(df['ts'].dt.date).size()
# plt.figure()
# grp.plot(kind='line')  # NO especifiques colores
# plt.title("Solicitudes por día")
# plt.xlabel("Día")
# plt.ylabel("Conteo")
# save_figure("req_por_dia.png")

# Repite con heatmap hora×día, barras apiladas, rankings, histograma, etc.



---
## Etapa D — Detección de patrones y campañas (reglas + evidencias)

Genera `resultados/eventos_flagged.csv` con cabecera:  
`event_id,ts,src_ip,host,url_path,status,attack_type,rule,evidence,severity`

**Campañas a detectar (una función por campaña):**
- Scanner 404/403 diario (~03:05).
- Fuerza bruta a WordPress (~14:10; domingo 00:00 extra).
- SQLi (lun/mié/vie ~16:20).
- LFI/Traversal (mar/jue ~05:45).
- XSS (diario ~12:30).
- Command injection (días alternos ~18:05).
- Rate limiting 429 (diario 10:00).
- Micro‑DDoS (21‑Mar 18:47–18:48).

**Estructura sugerida:** cada función devuelve **un DataFrame** con las columnas requeridas.


In [ ]:

# === Esqueletos de detección (rellena con tu lógica) ===
# Sugerencia: usa ventanas temporales (resample a '1T' o '1S'), groupby por IP/host, y umbrales.

def detect_scanner(df: pd.DataFrame) -> pd.DataFrame:
    """Regla ejemplo (personaliza):
    - ≥ N peticiones con 404/403 a endpoints sensibles desde la misma IP en ≤ 2 minutos.
    Devuelve DataFrame con columnas requeridas.
    """
    raise NotImplementedError


def detect_bruteforce_wp(df: pd.DataFrame) -> pd.DataFrame:
    """Detecta ráfagas POST a /wp-login.php con mayoría 401 desde misma IP→host,
    marcando si culmina en 302/200.
    """
    raise NotImplementedError


def detect_sqli(df: pd.DataFrame) -> pd.DataFrame:
    """Patrones en query_string (UNION SELECT, ' OR '1'='1, --), cruzado con WAF SQLI-...."""
    raise NotImplementedError


def detect_lfi_traversal(df: pd.DataFrame) -> pd.DataFrame:
    """Patrones ../, %2e%2e y rutas sensibles; valida con reglas LFI/TRAVERSAL."""
    raise NotImplementedError


def detect_xss(df: pd.DataFrame) -> pd.DataFrame:
    """Patrones <script>, onerror=...; contrasta user-agent/host y WAF XSS-...."""
    raise NotImplementedError


def detect_cmdi(df: pd.DataFrame) -> pd.DataFrame:
    """Patrones ';cat /etc/passwd', 'cmd='...; valida con WAF CMDI-...."""
    raise NotImplementedError


def detect_rate_limit(df: pd.DataFrame) -> pd.DataFrame:
    """Picos de 429 en /api/v1/items desde misma IP; calcula ratio 429/total por día."""
    raise NotImplementedError


def detect_micro_ddos(df: pd.DataFrame) -> pd.DataFrame:
    """Micro-brote de 60–120s con req/s altos a mismo host desde múltiples IPs
    y aumento de 5xx/429; devuelve eventos representativos del intervalo.
    """
    raise NotImplementedError


In [ ]:

# TODO: Consolidación de detecciones y guardado a CSV
# detections = []
# detections += [detect_scanner(df)]
# detections += [detect_bruteforce_wp(df)]
# detections += [detect_sqli(df)]
# detections += [detect_lfi_traversal(df)]
# detections += [detect_xss(df)]
# detections += [detect_cmdi(df)]
# detections += [detect_rate_limit(df)]
# detections += [detect_micro_ddos(df)]

# from pandas import concat
# flagged = concat(detections, ignore_index=True)
# out_csv = OUT_DIR / "eventos_flagged.csv"
# flagged.to_csv(out_csv, index=False)
# print("Guardado:", out_csv)



---
## Etapa E — Validación y métricas

**Cruces sugeridos:**  
- `waf_action`/`waf_rule_id` vs tus reglas (por campaña).  
- `ua_family` y `status_class` por campaña.  
- Métricas por campaña: `#eventos`, `#IPs`, `hosts`, `% WAF bloqueos`.

**Pregunta clave:** ¿qué podría generar **falsos positivos** o **falsos negativos** en tus reglas? ¿Cómo lo mitigarías?


In [ ]:

# TODO: Cálculo de métricas/resúmenes de validación por campaña
# Ejemplo de estructura (no resuelto):
# summary = (flagged.groupby('attack_type')
#                    .agg(eventos=('event_id','count'),
#                         ips=('src_ip','nunique'),
#                         hosts=('host','nunique'))
#           )
# print(summary)



---
## Etapa F — Informe (máx. 5 páginas, PDF)

**Estructura sugerida:**
1. **Resumen ejecutivo** (≤ 10 líneas).
2. **Metodología de limpieza** (decisiones y su impacto).
3. **EDA** (2–3 insights apoyados con figuras).
4. **Detecciones por campaña**: 1 subsección por campaña con **regla**, **métrica** y **gráfico**.
5. **Conclusiones y acciones recomendadas** (WAF, rate limit, monitoreo, *honeypots*, etc.).
6. **(Opcional)** Idea de IA para futuras mejoras (features por IP/día, IsolationForest/DBSCAN, validación).

> Exporta las figuras en `resultados/figuras/` y referencia los nombres en el PDF.



---
## Checklist de entrega

- [ ] Notebook ejecuta sin errores (una vez completado).
- [ ] Figuras guardadas en `resultados/figuras/` con títulos/ejes/leyendas legibles.
- [ ] `resultados/eventos_flagged.csv` con columnas: `event_id,ts,src_ip,host,url_path,status,attack_type,rule,evidence,severity`.
- [ ] Informe PDF (≤ 5 páginas) con narrativa clara y evidencias.
- [ ] `requirements.txt` o `environment.yml`.
- [ ] Estructura de carpetas respetada.
